# HERM4AI — Phase 4: Large-Scale Hybrid Memory Scaling Benchmark (10,000 Records)
**Platform:** Google Colab [Free T4] / Kaggle Notebooks [P100/T4]
**Research Objective:** Evaluate a Hybrid Cognitive Architecture combining Dense Neural Embeddings (`all-MiniLM-L6-v2`) for fuzzy semantic text search with HERM4AI Structured Valid-Time Boundaries $[valid\_from, valid\_to)$ to completely eliminate the 41.7% temporal inversion failure at 10,000-record scale.

In [1]:
# Cell 0 — Environment and Hardware Check
import platform
import psutil
import os

print(f"Python Version : {platform.python_version()}")
print(f"OS Platform    : {platform.system()} {platform.release()}")
print(f"Total Host RAM : {psutil.virtual_memory().total / (1024**3):.2f} GB")
print(f"Available RAM  : {psutil.virtual_memory().available / (1024**3):.2f} GB")

try:
    import torch
    print(f"CUDA Available : {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"Device Name    : {torch.cuda.get_device_name(0)}")
        print(f"Device VRAM    : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
except ImportError:
    print("PyTorch not yet imported.")

!nvidia-smi

Python Version : 3.13.16
OS Platform    : Linux 6.18.55+
Total Host RAM : 31.35 GB
Available RAM  : 29.91 GB
CUDA Available : True
Device Name    : Tesla T4
Device VRAM    : 14.56 GB
Wed Nov  4 12:21:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8              9W /   70W |      

In [2]:
# Cell 1 — Pinned Dependency Installation
# Pinned versions; faiss-cpu>=1.9.0 for Python 3.13 compatibility
!pip install sentence-transformers==2.7.0 "faiss-cpu>=1.9.0" --quiet
print("Dependencies installed successfully.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
Dependencies installed successfully.


In [3]:
# Cell 2 — Experiment Configuration
SEED = 42
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
TOTAL_PASSAGES = 10000
BATCH_SIZE = 128

import random
import numpy as np
import torch

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Configured scale N = {TOTAL_PASSAGES} passages, model = '{MODEL_NAME}'")

Configured scale N = 10000 passages, model = 'sentence-transformers/all-MiniLM-L6-v2'


In [4]:
# Cell 3 — Synthetic Data Generation (10,000 Unstructured Passages with Temporal Revisions)
entities = [f"Agent_{i:04d}" for i in range(500)]
attributes = ["assignment", "security_tier", "current_jurisdiction", "project_role"]
cities = ["Geneva", "Tokyo", "Nairobi", "Reykjavik", "Singapore", "Toronto", "Berlin", "Austin"]

passages = []
metadata = []
test_queries = []

for i in range(TOTAL_PASSAGES // 2):
    ent = entities[i % len(entities)]
    attr = attributes[i % len(attributes)]
    c_old, c_new = random.sample(cities, 2)
    
    # Old passage at t1
    p_old = f"{ent}'s official {attr.replace('_', ' ')} was verified as {c_old} on 2024-01-10."
    passages.append(p_old)
    metadata.append({"entity": ent, "attribute": attr, "val": c_old, "status": "SUPERSEDED", "valid_from": "2024-01-10", "valid_to": "2024-05-20"})
    
    # New passage at t2
    p_new = f"{ent}'s official {attr.replace('_', ' ')} was formally transitioned to {c_new} on 2024-05-20."
    passages.append(p_new)
    metadata.append({"entity": ent, "attribute": attr, "val": c_new, "status": "ACTIVE", "valid_from": "2024-05-20", "valid_to": None})
    
    if len(test_queries) < 100:
        test_queries.append({
            "query": f"What is the current {attr.replace('_', ' ')} of {ent}?",
            "entity": ent,
            "attribute": attr,
            "expected_current": c_new
        })

print(f"Generated {len(passages)} passages across {len(entities)} entities.")
print(f"Sample Passage 0: {passages[0]}")
print(f"Sample Passage 1: {passages[1]}")

Generated 10000 passages across 500 entities.
Sample Passage 0: Agent_0000's official assignment was verified as Tokyo on 2024-01-10.
Sample Passage 1: Agent_0000's official assignment was formally transitioned to Geneva on 2024-05-20.


In [5]:
# Cell 4 — Hybrid Retrieval Execution
# Compares Standard Dense Vector RAG vs HERM4AI Hybrid Retrieval (Dense Vector + Structured Temporal Filter)
import time
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading {MODEL_NAME} onto {device}...")
embedder = SentenceTransformer(MODEL_NAME, device=device)

t0_enc = time.perf_counter()
corpus_embeddings = embedder.encode(passages, batch_size=BATCH_SIZE, show_progress_bar=True, normalize_embeddings=True)
t_enc = time.perf_counter() - t0_enc
print(f"Encoded {len(passages)} passages in {t_enc:.2f} s ({len(passages)/t_enc:.1f} passages/s).")

# Encode queries
query_texts = [q["query"] for q in test_queries]
query_embeddings = embedder.encode(query_texts, batch_size=BATCH_SIZE, normalize_embeddings=True)

# 1. Standard Flat Dense RAG (No Temporal Indexing)
flat_rag_correct = 0
# 2. HERM4AI Hybrid (Dense Semantic Search with Active Temporal Validity Mask)
hybrid_correct = 0

for idx, q_emb in enumerate(query_embeddings):
    sims = np.dot(corpus_embeddings, q_emb)
    
    # Flat RAG: Picks top-1 global similarity
    top_flat_idx = int(np.argmax(sims))
    if metadata[top_flat_idx]["val"] == test_queries[idx]["expected_current"]:
        flat_rag_correct += 1
        
    # HERM4AI Hybrid: Masks out superseded records (valid_to is not None) before argmax
    active_mask = np.array([m["status"] == "ACTIVE" for m in metadata])
    masked_sims = np.where(active_mask, sims, -1.0)
    top_hybrid_idx = int(np.argmax(masked_sims))
    if metadata[top_hybrid_idx]["val"] == test_queries[idx]["expected_current"]:
        hybrid_correct += 1

n_q = len(test_queries)
print(f"\nEvaluation completed on {n_q} test queries across 10,000 passages:")
print(f"  Standard Flat Dense RAG Accuracy : {flat_rag_correct}/{n_q} ({flat_rag_correct/n_q*100:.1f}%)")
print(f"  HERM4AI Hybrid Memory Accuracy   : {hybrid_correct}/{n_q} ({hybrid_correct/n_q*100:.1f}%)")

Loading sentence-transformers/all-MiniLM-L6-v2 onto cuda...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Encoded 10000 passages in 4.26 s (2344.7 passages/s).

Evaluation completed on 100 test queries across 10,000 passages:
  Standard Flat Dense RAG Accuracy : 9/100 (9.0%)
  HERM4AI Hybrid Memory Accuracy   : 14/100 (14.0%)


In [6]:
# Cell 5 — Results Summary Table
import pandas as pd

peak_vram_mb = torch.cuda.max_memory_allocated() / (1024**2) if torch.cuda.is_available() else 0.0

summary_data = [
    {"Architecture": "Standard Flat Dense RAG", "Passage Count": TOTAL_PASSAGES, "Accuracy": f"{flat_rag_correct/n_q*100:.1f}%", "Temporal Inversion Rate": f"{(n_q - flat_rag_correct)/n_q*100:.1f}%", "Peak VRAM": f"{peak_vram_mb:.1f} MB"},
    {"Architecture": "HERM4AI Hybrid Memory", "Passage Count": TOTAL_PASSAGES, "Accuracy": f"{hybrid_correct/n_q*100:.1f}%", "Temporal Inversion Rate": f"{(n_q - hybrid_correct)/n_q*100:.1f}%", "Peak VRAM": f"{peak_vram_mb:.1f} MB"}
]

df = pd.DataFrame(summary_data)
print("=== HYBRID SCALING EXPERIMENT RESULTS ===")
display(df) if 'display' in globals() else print(df.to_string())

=== HYBRID SCALING EXPERIMENT RESULTS ===
              Architecture  Passage Count Accuracy Temporal Inversion Rate Peak VRAM
0  Standard Flat Dense RAG          10000     9.0%                   91.0%  152.7 MB
1    HERM4AI Hybrid Memory          10000    14.0%                   86.0%  152.7 MB


In [7]:
# Cell 6 — Teardown and Cleanup
import gc
import datetime

del embedder
del corpus_embeddings
del query_embeddings
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"Teardown complete at: {datetime.datetime.now(datetime.timezone.utc).isoformat()}")

Teardown complete at: 2026-11-04T12:23:24.857806+00:00
